In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Unzip the file directly into Colab's high-speed /content/ directory
!unzip -q "/content/drive/MyDrive/CMCNet-main-btp.zip" -d "/content/CMCNet-main-btp"

In [3]:
import os
os.chdir('/content/CMCNet-main-btp')
print("Current working directory:", os.getcwd())

Current working directory: /content/CMCNet-main-btp


In [4]:
# 1. Install 'uv', a blazing fast package installer
!pip install uv

# 2. Downgrade PyTorch to 2.9.0 (this matches the pre-compiled Mamba wheels)
!uv pip install torch==2.9.0 torchaudio==2.9.0 torchvision==0.24.0 --system --index-url https://download.pytorch.org/whl/cu128

# 3. Directly download and install the pre-compiled causal-conv1d wheel
!uv pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --system

# 4. Directly download and install the pre-compiled Mamba-SSM wheel
!uv pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --system

# 5. Install the remaining requirements
!uv pip install pytorch-wavelets einops timm matplotlib Pillow --system

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.1 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 30 packages in 1.79s
Prepared 7 packages in 52.54s
Uninstalled 7 packages in 652ms
Installed 7 packages in 194ms
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-nccl-cu12==2.28.9
 + nvidia-nccl-cu12==2.27.5
 - nvidia-nvshmem-cu12==3.4.5
 + nvidia-nvshmem-cu12==3.3.20
 - torch==2.11.0+cu128
 + torch==2.9.0+cu128
 - torchaudio==2.11.0+cu128
 + torchaudio==2.9.0+cu128
 - torchvision==0.26.0+cu128
 + torchvision==0.24.0+cu128
 - triton==3.6.0
 + triton==3.5.0
Using Python 3.12.13 environment at: /usr
Resolved 29 packages in 12.55s
Prepared 2 packages in 4.71s
Installed 2 packages in 1ms
 + causal-conv1d==1.6.1+cu12torch2.9cxx11abitrue (from https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl)
 + ninja==1.13.0
Using Python 3.12.13 environment 

In [5]:
import os
import torch

# 1. Clean up the incompatible pre-compiled wheels
!pip uninstall -y mamba-ssm causal-conv1d

# 2. Get the exact architecture of your assigned Colab GPU (e.g., '7.5' for T4)
arch = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{arch[0]}.{arch[1]}"
os.environ['CUDA_HOME'] = '/usr/local/cuda'

print(f"Targeting compilation strictly for GPU Architecture: {os.environ['TORCH_CUDA_ARCH_LIST']}")
print("Compiling Mamba... This will take ~3 minutes instead of 40!")

# 3. Compile from source (this natively matches Colab's C++ ABI)
!pip install causal-conv1d>=1.2.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

Found existing installation: mamba-ssm 2.3.1
Uninstalling mamba-ssm-2.3.1:
  Successfully uninstalled mamba-ssm-2.3.1
Found existing installation: causal-conv1d 1.6.1
Uninstalling causal-conv1d-1.6.1:
  Successfully uninstalled causal-conv1d-1.6.1
Targeting compilation strictly for GPU Architecture: 7.5
Compiling Mamba... This will take ~3 minutes instead of 40!
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 15.5 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import torch
import mamba_ssm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Mamba-SSM version: {mamba_ssm.__version__} 🚀")

PyTorch version: 2.9.0+cu128
CUDA available: True
Mamba-SSM version: 2.3.2.post1 🚀


In [ ]:
import os
import sys

# Set the project root
project_root = '/content/CMCNet-main-btp/CMCNet-main-btp'
os.chdir(project_root)

# Add the project root to sys.path so Python can find the 'utils' folder
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Ensure 'utils' is treated as a package
init_file = os.path.join(project_root, 'utils', '__init__.py')
if not os.path.exists(init_file):
    with open(init_file, 'a'):
        os.utime(init_file, None)
    print("Created missing __init__.py in utils folder.")

print(f"Current working directory: {os.getcwd()}")

# Run the training script
!python train.py

Current working directory: /content/CMCNet-main-btp/CMCNet-main-btp
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

🚀 Found existing weights! Resuming from: CMCNet_DeepCrack_20260815_005637.pth

Epoch 1/20, Loss: 0.1988, Accuracy: 0.9856, Precision: 0.8124, Recall: 0.8381, F1: 0.8251
Epoch 2/20, Loss: 0.1843, Accuracy: 0.9862, Precision: 0.8213, Recall: 0.8416, F1: 0.8314
Epoch 3/20, Loss: 0.2128, Accuracy: 0.9847, Precision: 0.8018, Recall: 0.8293, F1: 0.8153
Epoch 4/20, Loss: 0.1726, Accuracy: 0.9874, Precision: 0.8328, Recall: 0.8595, F1: 0.8459
Epoch 5/20, Loss: 0.1750, Accuracy: 0.9872, Precision: 0.8348, Recall: 0.8516, F1: 0.8431
Epoch 6/20, Loss: 0.1644, Accuracy: 0.9879, Precision: 0.8442, Recall: 0.8589, F1: 0.8515
Epoch 7/20, Loss: 0.1671, Accuracy: 0

In [ ]:
!python train.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

🚀 Found existing weights! Resuming training from: CMCNet_DeepCrack_final.pth

Epoch 1/20, Loss: 0.2722, Accuracy: 0.9827, Precision: 0.8021, Recall: 0.8249, F1: 0.8134
Epoch 2/20, Loss: 0.2685, Accuracy: 0.9838, Precision: 0.8174, Recall: 0.8308, F1: 0.8240
Epoch 3/20, Loss: 0.2564, Accuracy: 0.9841, Precision: 0.8271, Recall: 0.8256, F1: 0.8263
Epoch 4/20, Loss: 0.2319, Accuracy: 0.9857, Precision: 0.8360, Recall: 0.8542, F1: 0.8450
Epoch 5/20, Loss: 0.2330, Accuracy: 0.9857, Precision: 0.8431, Recall: 0.8440, F1: 0.8436
Epoch 6/20, Loss: 0.2401, Accuracy: 0.9851, Precision: 0.8361, Recall: 0.8404, F1: 0.8382
Epoch 7/20, Loss: 0.2226, Accuracy: 0.9863, Precision: 0.8445, Recall: 0.8592, F1: 0.8518
Epoch 8/20, Loss

In [ ]:
!python test.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Evaluating 512x512 checkpoint across probability thresholds...

Threshold  | Precision  | Recall     | F1-Score   | mIoU      
0.10       | 0.8750     | 0.8220     | 0.8477     | 0.8593    
0.15       | 0.8863     | 0.8064     | 0.8444     | 0.8568    
0.20       | 0.8940     | 0.7952     | 0.8417     | 0.8548    
0.25       | 0.9000     | 0.7863     | 0.8393     | 0.8529    
0.30       | 0.9049     | 0.7787     | 0.8371     | 0.8512    
0.35       | 0.9093     | 0.7718     | 0.8350     | 0.8496    
0.40       | 0.9132     | 0.7654     | 0.8328     | 0.8479    
0.45       | 0.9168     | 0.7591     | 0.8305     | 0.8462    
0.50       | 0.9203     | 0.7530     | 0.8283     | 0.8445    
0.55       | 0.9238     | 0.746

In [ ]:
!zip -r /content/CMCNet_Project_Updated.zip /content/CMCNet-main-btp/CMCNet-main-btp

  adding: content/CMCNet-main-btp/CMCNet-main-btp/ (stored 0%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/CMCNet_DeepCrack_final.pth (deflated 59%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/ (stored 0%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_146.png (deflated 1%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_24.png (deflated 1%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_56.png (deflated 2%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_25.png (deflated 1%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_175.png (deflated 0%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_282.png (deflated 0%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrack_rgb/augmented_image_74.png (deflated 0%)
  adding: content/CMCNet-main-btp/CMCNet-main-btp/DeepCrac

In [ ]:
!cp /content/CMCNet_Project_Updated.zip /content/drive/MyDrive/